**Assessment 1: Mini RAG Pipeline with ChromaDB on Colab**


# Install libraries

In [27]:
!pip -q install chromadb sentence-transformers

# Imports


In [28]:
import chromadb
import string
from sentence_transformers import SentenceTransformer

In [29]:
documents = [
    "Artificial Intelligence is transforming healthcare.",
    "Machine Learning helps computers learn from data.",
    "Natural Language Processing enables computers to understand text.",
    "Deep Learning is a subset of Machine Learning.",
    "Computer Vision allows machines to interpret images."
]

#Preprocessing

In [30]:
def preprocess(text):
    text = text.lower()
    text = text.translate(str.maketrans('', '', string.punctuation))
    return text

processed_docs = [preprocess(doc) for doc in documents]

processed_docs

['artificial intelligence is transforming healthcare',
 'machine learning helps computers learn from data',
 'natural language processing enables computers to understand text',
 'deep learning is a subset of machine learning',
 'computer vision allows machines to interpret images']

In [31]:
model = SentenceTransformer("all-MiniLM-L6-v2")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

#Load embedding model

In [32]:
embeddings = model.encode(processed_docs).tolist()

# ChromaDB (mini_rag)

In [33]:
client = chromadb.Client()

collection = client.get_or_create_collection("mini_rag")

collection.add(
    documents=processed_docs,
    embeddings=embeddings,
    ids=[str(i) for i in range(len(processed_docs))]
)

# Query

In [34]:
query = input("Enter your query: ")

query = preprocess(query)

query_embedding = model.encode(query).tolist()

Enter your query: what is NLP ?


In [35]:
results = collection.query(
    query_embeddings=[query_embedding],
    n_results=1
)

print("Most Relevant Document:")
print(results["documents"][0][0])

Most Relevant Document:
natural language processing enables computers to understand text


# End of Assessment 1

**Assessment 2: Building a Knowledge Graph with Semantic Search and Query Answering**

In [36]:
!pip -q install rdflib chromadb sentence-transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 615.4/615.4 kB 10.0 MB/s eta 0:00:00


In [37]:
from rdflib import Graph, Namespace, RDF
import chromadb
from sentence_transformers import SentenceTransformer

#RDF Graph

In [38]:
g = Graph()

EX = Namespace("http://example.org/")

g.add((EX.Alice, RDF.type, EX.Person))
g.add((EX.Alice, EX.knows, EX.Bob))
g.add((EX.Bob, EX.worksAt, EX.AcmeCorp))
g.add((EX.Bob, RDF.type, EX.Person))

<Graph identifier=Nd37707f2f1fb4b57b2deed6d74e926f3 (<class 'rdflib.graph.Graph'>)>

# Convert triples to text

In [39]:
sentences = []

for s, p, o in g:
    subject = s.split("/")[-1]
    predicate = p.split("/")[-1]
    obj = o.split("/")[-1]

    sentence = f"{subject} {predicate} {obj}"
    sentences.append(sentence)

sentences

['Alice knows Bob',
 'Bob worksAt AcmeCorp',
 'Bob 22-rdf-syntax-ns#type Person',
 'Alice 22-rdf-syntax-ns#type Person']

In [41]:
model = SentenceTransformer("all-MiniLM-L6-v2")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [42]:
embeddings = model.encode(sentences).tolist()

# ChromaDB (knowledge_graph)

In [43]:
client = chromadb.Client()

collection = client.create_collection("knowledge_graph")

collection.add(
    documents=sentences,
    embeddings=embeddings,
    ids=[str(i) for i in range(len(sentences))]
)

In [45]:
query = input("Ask a question: ")

query_embedding = model.encode(query).tolist()

Ask a question: who does alice know?


In [46]:
results = collection.query(
    query_embeddings=[query_embedding],
    n_results=1
)

print("Relevant Knowledge:")
print(results["documents"][0][0])

Relevant Knowledge:
Alice knows Bob
